# Prévision à une semaine des hospitalisations COVID-19

Dans le notebook précédent, chaque modèle prévoyait les 54 semaines de la période de test en une seule fois, à partir de la fin de l'entraînement. Ici, on se place dans une situation plus proche de l'usage réel : chaque semaine, on prévoit uniquement la semaine suivante, en connaissant toutes les valeurs observées jusqu'à aujourd'hui.

Les paramètres de chaque modèle sont estimés une seule fois sur la période d'entraînement, puis ils restent fixes. À chaque nouvelle semaine du test, le modèle intègre la vraie valeur observée et fait une nouvelle prévision pour la semaine d'après. Les erreurs obtenues ne sont donc pas comparables à celles de la prévision sur 54 semaines : il s'agit d'une évaluation différente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

## 1. Données et découpage

On reprend la même série et le même découpage que dans le notebook précédent : entraînement de mars 2020 à fin septembre 2025, test d'octobre 2025 à octobre 2026.

In [ ]:
df = pd.read_csv("hospitalisations_hebdo_clean.csv", parse_dates=["date"], index_col="date")
y = df["covid"].asfreq("W-SAT")

TEST_START = pd.Timestamp("2025-10-04")
y_train = y[y.index < TEST_START]
y_test = y[y.index >= TEST_START]
H = len(y_test)

print(f"Entraînement : {len(y_train)} semaines")
print(f"Test : {H} semaines")

In [ ]:
def rmse(vrai, prevu):
    return np.sqrt(np.mean((np.asarray(vrai) - np.asarray(prevu)) ** 2))

def mae(vrai, prevu):
    return np.mean(np.abs(np.asarray(vrai) - np.asarray(prevu)))

previsions = {}

def comparer(nom, prev):
    prev = pd.Series(np.asarray(prev), index=y_test.index)
    previsions[nom] = prev

    plt.figure(figsize=(12, 4))
    plt.plot(y[-80:], color="black", lw=2, label="Observé")
    plt.plot(previsions["Naïf"], color="grey", ls="--", label="Naïf")
    plt.plot(previsions["Naïf saisonnier"], color="tab:orange", ls=":", lw=2, label="Naïf saisonnier")
    plt.plot(prev, color="tab:blue", lw=2, label=nom)
    plt.axvline(TEST_START, color="grey", lw=0.8)
    plt.title(f"{nom} : prévision à une semaine")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    for n in ["Naïf", "Naïf saisonnier", nom]:
        print(f"{n:<40} RMSE = {rmse(y_test, previsions[n]):.2f}   MAE = {mae(y_test, previsions[n]):.2f}")

## 2. Baselines

À une semaine, la prévision naïve consiste à dire que la semaine prochaine sera égale à la semaine actuelle. La prévision naïve saisonnière reprend la valeur de la même semaine un an plus tôt.

Le naïf devient ici une référence beaucoup plus difficile à battre que dans la prévision sur 54 semaines. La série évolue lentement d'une semaine à l'autre, donc la valeur de la semaine précédente est déjà une très bonne estimation. Un modèle n'a d'intérêt que s'il anticipe mieux les montées et les descentes.

In [ ]:
previsions["Naïf"] = y.shift(1).loc[y_test.index]
previsions["Naïf saisonnier"] = y.shift(52).loc[y_test.index]

plt.figure(figsize=(12, 4))
plt.plot(y[-80:], color="black", lw=2, label="Observé")
plt.plot(previsions["Naïf"], color="grey", ls="--", label="Naïf")
plt.plot(previsions["Naïf saisonnier"], color="tab:orange", ls=":", lw=2, label="Naïf saisonnier")
plt.axvline(TEST_START, color="grey", lw=0.8)
plt.title("Baselines : prévision à une semaine")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for n in ["Naïf", "Naïf saisonnier"]:
    print(f"{n:<20} RMSE = {rmse(y_test, previsions[n]):.2f}   MAE = {mae(y_test, previsions[n]):.2f}")

Le naïf obtient une RMSE de 0.16 et une MAE de 0.11. En moyenne, la valeur de la semaine précédente ne se trompe que d'environ 0.1 point de taux. Le naïf saisonnier est beaucoup moins bon (RMSE de 1.18, MAE de 1.11), car il recopie l'année précédente, dont les pics étaient bien plus élevés. Les modèles devront donc faire mieux qu'une erreur moyenne de 0.11 pour être utiles.

## 3. Méthode pour les modèles ARIMA, SARIMA et SARIMAX

On estime le modèle sur l'entraînement, puis on applique les paramètres obtenus à toute la série (entraînement et test) avec `apply`. Le modèle n'est pas ré-estimé : il utilise simplement les vraies valeurs passées pour mettre à jour son état. Ensuite, `get_prediction` donne, pour chaque semaine du test, la prévision faite à partir des données disponibles la semaine précédente.

La fonction ci-dessous regroupe ces étapes.

In [ ]:
def prevision_une_semaine(order, seasonal_order=(0, 0, 0, 0), exog=None):
    exog_train = exog.loc[y_train.index] if exog is not None else None

    res = SARIMAX(y_train, order=order, seasonal_order=seasonal_order, exog=exog_train).fit(disp=False)
    res_complet = res.apply(y, exog=exog)
    prev = res_complet.get_prediction(start=y_test.index[0], end=y_test.index[-1]).predicted_mean

    return prev

## 4. ARIMA

In [ ]:
comparer("ARIMA(2,1,0)", prevision_une_semaine((2, 1, 0)))

Résultats à compléter.

In [ ]:
comparer("ARIMA(0,1,2)", prevision_une_semaine((0, 1, 2)))

Résultats à compléter.

Contrairement à la prévision sur 54 semaines, la prévision d'un ARIMA à une semaine n'est pas plate. Le modèle connaît toujours les dernières variations réelles, donc il peut suivre la série. La question est de savoir s'il anticipe mieux que le naïf, qui se contente de recopier la dernière valeur.

## 5. SARIMA

In [ ]:
comparer("SARIMA(0,1,2)(1,0,0)52", prevision_une_semaine((0, 1, 2), (1, 0, 0, 52)))

Résultats à compléter.

In [ ]:
comparer("SARIMA(0,1,2)(0,1,1)52", prevision_une_semaine((0, 1, 2), (0, 1, 1, 52)))

Résultats à compléter. Sur 54 semaines, ce modèle surestimait fortement le pic. À une semaine, il est corrigé chaque semaine par la vraie valeur observée, ce qui devrait limiter ce problème.

## 6. SARIMAX avec des termes de Fourier

In [ ]:
PERIODE = 365.25 / 7
t = np.arange(len(y))

def fourier(K):
    X = pd.DataFrame(index=y.index)
    for k in range(1, K + 1):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * t / PERIODE)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * t / PERIODE)
    return X

comparer("SARIMAX(0,1,2) + Fourier K=3", prevision_une_semaine((0, 1, 2), exog=fourier(3)))

Résultats à compléter.

## 7. Holt-Winters

La bibliothèque ne permet pas d'appliquer directement des paramètres fixes à de nouvelles données pour Holt-Winters. On procède donc autrement : chaque semaine du test, on ré-estime le modèle sur toutes les données disponibles jusqu'à la veille, puis on prévoit la semaine suivante. Cette méthode est un peu plus lente, car le modèle est estimé 54 fois.

In [ ]:
def holt_winters_une_semaine(saison):
    prev = []
    for date in y_test.index:
        historique = y[y.index < date]
        hw = ExponentialSmoothing(
            historique, trend="add", damped_trend=True, seasonal=saison, seasonal_periods=52
        ).fit()
        prev.append(hw.forecast(1).iloc[0])
    return prev

comparer("Holt-Winters additif", holt_winters_une_semaine("add"))

Résultats à compléter.

In [ ]:
comparer("Holt-Winters multiplicatif", holt_winters_une_semaine("mul"))

Résultats à compléter.

## 8. Comparaison de tous les modèles

In [ ]:
resultats = pd.DataFrame({
    nom: {"RMSE": rmse(y_test, prev), "MAE": mae(y_test, prev)}
    for nom, prev in previsions.items()
}).T.sort_values("MAE").round(2)

resultats

## 9. Conclusion

Conclusion à compléter avec les résultats. Les points à discuter sont les suivants : quels modèles battent le naïf à une semaine, si le classement des modèles est le même que pour la prévision sur 54 semaines, et si les modèles qui surestimaient fortement la saison (SARIMA avec différence saisonnière, Fourier) deviennent acceptables lorsqu'ils sont corrigés chaque semaine par les vraies valeurs.